# CSA paper — kiểm tra core tự viết

Bật **Internet** để tải source đã khóa SHA. Chọn **GPU T4**; chỉ dùng GPU0,
không cần hai GPU, dataset, checkpoint, API token, spaCy hoặc tải CLIP.
Save Version → Save & Run All. Không tự chạy notebook bằng agent.

Kiểm tensor nhỏ, rồi D512/8 heads/K150/3 decoder layers/vocab10000:
2 cập nhật XE + 1 cập nhật RL với dữ liệu và reward **giả**. Đây là kiểm mã,
không phải huấn luyện thực nghiệm; không có CIDEr/CHAIR. Thời gian worker
tối đa30phút là giới hạn dừng, không phải dự đoán thời gian.

Gửi `csa_paper_core_smoke_review.zip` dù thành công hoặc lỗi.
Sau smoke mới kiểm detector/region cache và tạo bank TRAIN thực.


In [ ]:
import hashlib, json, os, queue, signal, subprocess, sys, tempfile, threading, time, traceback, urllib.request, zipfile
from pathlib import Path
COMMIT = '6c365d3f9f57da3f69cd43244e7b5fd2d51af128'
HASHES = {'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'test_data.py': '0a5b114ee4b3cc1b8a0a612ed144fd1b57bda39f792244421ed7c5f19a03521a', 'runtime_checks.py': '88e62c44f4b14d13962923b31049bd8399364813ca55624732b903a58b4f84f0', 'IMPLEMENTATION_PROTOCOL.md': '154774d6409f50b4f2c45b6b2dbd837624aefbbd3aacb9af72fbdad313defcfc'}
OUTPUT = Path('/kaggle/working/csa_paper_core_smoke')
if OUTPUT.exists():
    raise RuntimeError('Output exists. Preserve its review ZIP; use a fresh session for this smoke.')
OUTPUT.mkdir(parents=True)
SOURCE = Path(tempfile.mkdtemp(prefix='csa-paper-core-'))
LOG = OUTPUT/'notebook.log'
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT, sha256=HASHES, synthetic_only=True), indent=2), encoding='utf-8')
print('Source commit:', COMMIT, flush=True)
for name, expected in HASHES.items():
    url = f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}'
    payload = urllib.request.urlopen(url, timeout=90).read()
    if hashlib.sha256(payload).hexdigest() != expected: raise RuntimeError('Source hash differs: '+name)
    (SOURCE/name).write_bytes(payload)
print('Pinned source verified. No dataset, model weights or API token needed.', flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Smoke active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete = False
try:
    run([sys.executable, '-u', SOURCE/'test_data.py'], timeout=120)
    run([sys.executable, '-u', SOURCE/'runtime_checks.py', '--output', OUTPUT], timeout=1800)
    summary = json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
    complete = summary.get('status') == 'CORE_SMOKE_COMPLETE'
    if not complete: raise RuntimeError('Missing complete smoke status')
    print(json.dumps(summary, indent=2), flush=True)
except BaseException:
    (OUTPUT/'notebook_failure.txt').write_text(traceback.format_exc(), encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete, synthetic_only=True,
        quality_metrics_computed=False, real_data_verified=False, training_authorized=False)), encoding='utf-8')
    archive = OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file(): z.write(p, OUTPUT.name+'/'+str(p.relative_to(OUTPUT)))
    print('SEND REVIEW ZIP:', archive, flush=True)
    print('CORE SMOKE COMPLETE; not a quality result.' if complete else 'INCOMPLETE: send review ZIP/log.', flush=True)
